In [43]:
import pandas as pd
import numpy as np

In [46]:
# 1. Load dataset
df_raw = pd.read_csv('../data/raw/crop_data.csv')

In [ ]:
# 2. Standardize Column Names
df_raw.columns = (
    df_raw.columns
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

In [ ]:

# 3. Clean Text Columns
text_cols = ["State", "District", "Crop", "Season"]

for col in text_cols:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].astype(str).str.strip()

Reshaping the Data

In [28]:
# 2. Reshape (Melt) the 'Area' columns
df_area = df_raw.melt(id_vars=['State', 'District', 'Crop', 'Season'], 
                  value_vars=['Area-2022-23', 'Area-2023-24', 'Area-2024-25'],
                  var_name='Year_Type', value_name='Area')
df_area['Crop_Year'] = df_area['Year_Type'].str.replace('Area-', '')

# 3. Reshape (Melt) the 'Production' columns
df_prod = df_raw.melt(id_vars=['State', 'District', 'Crop', 'Season'], 
                  value_vars=['Production-2022-23', 'Production-2023-24', 'Production-2024-25'],
                  var_name='Year_Type', value_name='Production')

# 4. Reshape (Melt) the 'Yield' columns
df_yield = df_raw.melt(id_vars=['State', 'District', 'Crop', 'Season'], 
                   value_vars=['Yield-2022-23', 'Yield-2023-24', 'Yield-2024-25'],
                   var_name='Year_Type', value_name='Yield')

In [31]:
# 5. Combine them back together into the standard Long Format
df_long = df_area[['State', 'District', 'Crop', 'Season', 'Crop_Year', 'Area']].copy()
df_long['Production'] = df_prod['Production']
df_long['Yield'] = df_yield['Yield']

print("After Melting:", df_long.shape)
df_long.tail()

After Melting: (84807, 8)


,State,District,Crop,Season,Crop_Year,Area,Production,Yield
84802,West Bengal,Purulia,Total Pulses,Total,2024-25,NaN,NaN,NaN
84803,West Bengal,Purulia,Total Food Grains,Kharif,2024-25,NaN,NaN,NaN
84804,West Bengal,Purulia,Total Food Grains,Rabi,2024-25,NaN,NaN,NaN
84805,West Bengal,Purulia,Total Food Grains,Summer,2024-25,NaN,NaN,NaN
84806,West Bengal,Purulia,Total Food Grains,Total,2024-25,NaN,NaN,NaN


Cleaning the missing values

In [33]:
# Remove "Total" season rows
df_clean = df_long[df_long["Season"] != "Total"]

# Drop missing values
df_clean = df_clean.dropna(subset=["Area", "Production", "Yield"])

# Remove invalid Area
df_clean = df_clean[df_clean["Area"] > 0]

# Remove Production = 0 (important fix)
df_clean = df_clean[df_clean["Production"] > 0]

# Reset index
df_clean = df_clean.reset_index(drop=True)

print("After Cleaning:", df_clean.shape)
df_clean.head()

After Cleaning: (23755, 8)


,State,District,Crop,Season,Crop_Year,Area,Production,Yield
0,Andaman And Nicobar Islands,North And Middle Andaman,Rice,Kharif,2022-23,0.05,0.10,2110.0
1,Andaman And Nicobar Islands,North And Middle Andaman,Cereals,Kharif,2022-23,0.05,0.10,2110.0
2,Andaman And Nicobar Islands,North And Middle Andaman,Total Food Grains,Kharif,2022-23,0.05,0.10,2110.0
3,Andhra Pradesh,Alluri Sitharama Raju,Rice,Kharif,2022-23,0.57,1.69,2987.0
4,Andhra Pradesh,Alluri Sitharama Raju,Rice,Rabi,2022-23,0.02,0.06,2999.0


In [38]:
print(f"Rows before final filter: {len(df_clean)}")
print(df_clean.columns)
for col in df_clean.columns:
    print(f"\n{col}:")
    print(df_clean[col].unique())

Rows before final filter: 23755
Index(['State', 'District', 'Crop', 'Season', 'Crop_Year', 'Area',
       'Production', 'Yield'],
      dtype='str')

State:
<StringArray>
[                 'Andaman And Nicobar Islands',
                               'Andhra Pradesh',
                            'Arunachal Pradesh',
                                        'Assam',
                                        'Bihar',
                                   'Chandigarh',
                                 'Chhattisgarh',
                                          'Goa',
                                      'Gujarat',
                                      'Haryana',
                             'Himachal Pradesh',
                            'Jammu And Kashmir',
                                    'Jharkhand',
                                    'Karnataka',
                                       'Kerala',
                                       'Ladakh',
                               'Madhya Prades

In [41]:
df_clean = df_clean[df_clean['Season'].str.strip() != 'Total']
target_crops = [
    'Rice', 'Maize', 'Wheat', 'Ragi', 'Bajra', 'Jowar', 'Barley', 
    'Tur', 'Urad', 'Moong', 'Gram', 'Lentil'
]

df_focused = df_clean[df_clean['Crop'].str.strip().isin(target_crops)]


In [42]:
print("\nMissing Values:")
print(df_clean.isnull().sum())

print("\nZero Values in Numeric Columns:")
print(df_clean.select_dtypes(include="number").eq(0).sum())

print("\nBasic Statistics:")
df_clean.describe()


Missing Values:
State         0
District      0
Crop          0
Season        0
Crop_Year     0
Area          0
Production    0
Yield         0
dtype: int64

Zero Values in Numeric Columns:
Area          0
Production    0
Yield         0
dtype: int64

Basic Statistics:


,Area,Production,Yield
count,23755.000000,23755.000000,23755.000000
mean,0.454016,1.196613,2341.595369
std,0.742368,2.077742,1585.227066
min,0.010000,0.010000,15.000000
25%,0.030000,0.040000,1211.000000
50%,0.120000,0.230000,2078.000000
75%,0.600000,1.430000,3164.000000
max,12.480000,18.660000,44815.000000


In [13]:
print(f"Final model-ready rows: {len(df_focused)}")
print("\n--- Final Unique Crops in Dataset ---")
print(df_focused['Crop'].unique())
print("\n--- Final Unique Seasons in Dataset ---")
print(df_focused['Season'].unique())
print("\n--- Final Unique Crop Years in Dataset ---")
print(df_focused['Crop_Year'].unique())

Final model-ready rows: 11057

--- Final Unique Crops in Dataset ---
<StringArray>
[  'Rice',  'Maize',   'Ragi',    'Tur',   'Urad',  'Moong',  'Bajra',
  'Jowar',   'Gram',  'Wheat', 'Lentil', 'Barley']
Length: 12, dtype: str

--- Final Unique Seasons in Dataset ---
<StringArray>
['Kharif', 'Rabi', 'Summer']
Length: 3, dtype: str

--- Final Unique Crop Years in Dataset ---
<StringArray>
['2022-23', '2023-24', '2024-25']
Length: 3, dtype: str


In [37]:
df_clean[df_clean['Production'] == 0][['Area', 'Production', 'Yield']].head()

,Area,Production,Yield
